In [1]:
import pandas as pd
import os
from pathlib import Path

os.chdir ('C:\\Users\\hamod4691\\Code\\arctic-benthic-biodiversity\\OGSL')

In [2]:
# Add startDayOfYear, endDayOfYear, and year from event_date and save back to the same CSV

if "ogsl" not in globals():
    ogsl = pd.read_csv("Archambault_biodiversity_data_long_2026_03_11_CP.csv", sep=";")

event_dt = pd.to_datetime(ogsl["event_date"], errors="coerce")

ogsl["startDayOfYear"] = event_dt.dt.dayofyear.astype("Int32")
ogsl["endDayOfYear"] = event_dt.dt.dayofyear.astype("Int32")
ogsl["year"] = event_dt.dt.year.astype("Int32")

ogsl.to_csv("Archambault_biodiversity_data_long_2026_03_11_CP.csv", sep=";", index=False)

print("Saved Archambault_biodiversity_data_long_2026_03_11_CP.csv with startDayOfYear, endDayOfYear, and year.")

Saved Archambault_biodiversity_data_long_2026_03_11_CP.csv with startDayOfYear, endDayOfYear, and year.


# Prepare for merging

In [3]:
ogsl = pd.read_csv("Archambault_biodiversity_data_long_2026_03_11_CP.csv", sep=';')

dtype_spec = pd.read_excel(Path("..") / "column_data_types.xlsx", sheet_name=0)
dtype_map = dict(zip(dtype_spec['column'], dtype_spec['final']))

column_map = {
    "sourceID": "source_id",
    "datasetKey": "project_id",
    "decimalLatitudeStart": "lat_start",
    "decimalLongitudeStart": "lon_start",
    "decimalLatitudeStop": "lat_end",
    "decimalLongitudeStop": "lon_end",
    "depthA_m": "depth_start",
    "depthB_m": "depth_end",
    "eventID": "org_event_id_trawls",
    "locationID": "station_id",
    "verbatimDate": "event_date",
    "startDayOfYear": "startDayOfYear",
    "endDayOfYear": "endDayOfYear",
    "year": "year",
    "acceptedScientificName": "valid_scientific_name",
    "verbatimScientificName": "valid_scientific_name",
    "phylum": "phylum",
    "class": "class",
    "family": "family",
    "aphiaID": "valid_taxon_id",
    "individualCount": "density_ind_m2",
    "biomass": "biomass_g_m2",
    "gear": "sampling_protocol"
}

output_columns = [
    "ID",
    "sourceID",
    "verbatimID",
    "sourceDatabase",
    "datasetKey",
    "datasetTitle",
    "issue",
    "decimalLatitudeStart",
    "decimalLongitudeStart",
    "decimalLatitudeStop",
    "decimalLongitudeStop",
    "footprintWKT",
    "coordinateUncertaintyInMeters",
    "hasGeospatialIssues",
    "depthA_m",
    "depthAType",
    "depthB_m",
    "depthBType",
    "gebcoDepth",
    "eventID",
    "parentEventID",
    "waterBody",
    "largeMarineEcosystem",
    "locationID",
    "verbatimDate",
    "startDayOfYear",
    "endDayOfYear",
    "year",
    "eventRemarks",
    "verbatimScientificName",
    "acceptedScientificName",
    "identificationQualifier",
    "lifeStage",
    "sex",
    "taxonomicStatus",
    "taxonomicIssue",
    "taxonRank",
    "phylum",
    "class",
    "order",
    "family",
    "genus",
    "specificEpithet",
    "validAuthority",
    "aphiaID",
    "individualCount",
    "areaSampled",
    "areaSampledUnit",
    "volumeSampled_m3",
    "samplingEffort",
    "trawlingDistance_m",
    "biomass",
    "biomassUnit",
    "dynamicProperties",
    "basisOfRecord",
    "occurrenceRemarks",
    "gear",
    "gearCategory",
    "meshSize_mm",
    "sieveFraction_µm",
    "flagged",
    "flaggedComment", 
    "acceptedTaxonKey"
]


def normalize_dtype(dtype_name):
    dtype_name = str(dtype_name).strip().lower()
    return {
        "int32": "Int32",
        "float32": "Float32",
        "float64": "Float64",
        "boolean": "boolean",
        "string": "string",
    }.get(dtype_name, dtype_name)


def empty_series(index, dtype_name):
    dtype_name = normalize_dtype(dtype_name)
    if dtype_name in {"Int32", "Float32", "Float64", "boolean", "string"}:
        return pd.Series(pd.NA, index=index, dtype=dtype_name)
    return pd.Series(pd.NA, index=index, dtype="string")


def coerce_series(series, dtype_name):
    dtype_name = normalize_dtype(dtype_name)
    if dtype_name == "boolean":
        return series.astype("boolean")
    if dtype_name in {"Int32", "Float32", "Float64"}:
        return pd.to_numeric(series, errors="coerce").astype(dtype_name)
    return series.astype("string")


ogsl_final = pd.DataFrame(index=ogsl.index)
missing_sources = []

for col in output_columns:
    target_dtype = dtype_map.get(col, "string")
    if col in column_map:
        src = column_map[col]
        if src in ogsl.columns:
            ogsl_final[col] = coerce_series(ogsl[src], target_dtype)
        else:
            ogsl_final[col] = empty_series(ogsl.index, target_dtype)
            missing_sources.append(src)
    else:
        ogsl_final[col] = empty_series(ogsl.index, target_dtype)

ogsl_final = ogsl_final[output_columns]
ogsl_final.to_parquet("ogsl_final.parquet", index=False)

print(f"Saved ogsl_final.parquet with {len(ogsl_final):,} rows and {len(ogsl_final.columns)} columns.")
if missing_sources:
    print("Missing source columns in ogsl:", sorted(set(missing_sources)))

Saved ogsl_final.parquet with 7,257 rows and 63 columns.
Missing source columns in ogsl: ['source_id']


In [4]:
ogsl_final = pd.read_parquet("ogsl_final.parquet")
ogsl_final["sourceDatabase"] = "OGSL"
ogsl_final["gearCategory"] = "trawl"
ogsl_final.to_parquet("ogsl_final.parquet", index=False)
print("Added sourceDatabase column with value 'OGSL' and saved ogsl_final.parquet.")

Added sourceDatabase column with value 'OGSL' and saved ogsl_final.parquet.


In [5]:
ogsl_final = pd.read_parquet("ogsl_final.parquet")

ogsl_final.loc[ogsl_final["depthA_m"].notna(), "depthAType"] = "start"
ogsl_final.loc[ogsl_final["depthB_m"].notna(), "depthBType"] = "end"
ogsl_final.loc[ogsl_final["individualCount"].notna(), "areaSampled"] = 1.0
ogsl_final.loc[ogsl_final["individualCount"].notna(), "areaSampledUnit"] = "m2"
ogsl_final.loc[ogsl_final["biomass"].notna(), "biomassUnit"] = "g/m2"

ogsl_final.to_parquet("ogsl_final.parquet", index=False)
ogsl_final[["depthA_m", "depthAType", "depthB_m", "depthBType", "individualCount", "areaSampled", "areaSampledUnit", "biomass", "biomassUnit"]].head()

,depthA_m,depthAType,depthB_m,depthBType,individualCount,areaSampled,areaSampledUnit,biomass,biomassUnit
0,75.0,start,78.0,end,0.0036,1.0,m2,0.00042,g/m2
1,117.0,start,112.0,end,0.007199,1.0,m2,0.001742,g/m2
2,671.0,start,671.0,end,0.007679,1.0,m2,0.001014,g/m2
3,65.0,start,60.0,end,0.172784,1.0,m2,0.002315,g/m2
4,71.0,start,61.0,end,0.143,1.0,m2,0.109,g/m2


# Check data types

In [6]:
ogsl_final = pd.read_parquet("ogsl_final.parquet")

# Column type overview
column_types = pd.DataFrame({
    "column": ogsl_final.columns,
    "dtype": ogsl_final.dtypes.astype(str).values,
    "non_null_count": ogsl_final.notna().sum().values,
    "null_count": ogsl_final.isna().sum().values,
    "non_null_pct": (ogsl_final.notna().sum().values / len(ogsl_final)) * 100,
})

column_types = column_types.sort_values(
    by=["non_null_count", "column"], ascending=[False, True]
 )

column_types.to_csv("ogsl_final_column_types.csv", index=False)

column_types.head(20)

,column,dtype,non_null_count,null_count,non_null_pct
30,acceptedScientificName,string,7257,0,100.0
44,aphiaID,string,7257,0,100.0
4,datasetKey,string,7257,0,100.0
7,decimalLatitudeStart,Float64,7257,0,100.0
9,decimalLatitudeStop,Float64,7257,0,100.0
8,decimalLongitudeStart,Float64,7257,0,100.0
10,decimalLongitudeStop,Float64,7257,0,100.0
15,depthAType,string,7257,0,100.0
14,depthA_m,string,7257,0,100.0
17,depthBType,string,7257,0,100.0
